# unslop: Colab pipeline

Runtime → Change runtime type → **T4 GPU**. Add `GEMINI_API_KEY` and `ANTHROPIC_API_KEY` under the 🔑 *Secrets* sidebar and give this notebook access.

All data and the adapter are written to Google Drive (`RUN_DIR`), so a Colab disconnect costs you at most the current step. Judge calls are cached, so re-running a cell resumes it.

In [ ]:
# --- settings -------------------------------------------------------------
REPO_URL = "https://github.com/Akashkar00/unslop-rlhf.git"   # set to "" to use CODE_DIR on Drive instead
CODE_DIR = "/content/drive/MyDrive/unslop-rlhf"   # where the repo lives if REPO_URL is blank
RUN_DIR  = "/content/drive/MyDrive/unslop-run"    # data/, outputs/, reports/ go here
SMOKE    = False   # True = 0.5B model + 16 prompts, to check the plumbing cheaply

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

if REPO_URL:
    CODE_DIR = "/content/unslop-rlhf"
    if not os.path.exists(CODE_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)

for k in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):
    os.environ[k] = userdata.get(k)
os.environ["UNSLOP_ROOT"] = RUN_DIR
if SMOKE:
    os.environ["UNSLOP_BASE_MODEL"] = "Qwen/Qwen2.5-0.5B-Instruct"
LIMIT = "--limit 16" if SMOKE else ""
os.makedirs(RUN_DIR, exist_ok=True)
print("code:", CODE_DIR, "| run:", RUN_DIR, "| smoke:", SMOKE)

In [ ]:
%pip install -q -e "{CODE_DIR}[train]"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Prompts and candidates

In [ ]:
!unslop prompts
!unslop generate {LIMIT}

## 2. Preference pairs (train judge)

Prints position-consistency and how often the judge chose the longer response, *before* rebalancing. If `judge_chose_longer_share` is far above 0.5, the judge has a length bias, and the rebalancing step is doing real work.

In [ ]:
!unslop build-pairs {LIMIT}

## 3. DPO + LoRA

In [ ]:
!unslop train

## 4. Held-out eval

In [ ]:
!unslop generate-eval {LIMIT}
!unslop judge-eval
!unslop report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN_DIR}/reports/results.md").read()))

## 5. Human labels (on your laptop)

Download `RUN_DIR/data` from Drive into the repo's `data/` folder, then:

```bash
uv run streamlit run app/label_ui.py
uv run unslop report
```